# 00 – Vérification de l'environnement ShopStream
Exécutez toutes les cellules. Chaque étape doit afficher **OK**.

In [1]:
import os
from shopstream_utils import get_spark, read_table, write_table, DATA_DIR
print("Fichiers de données :", os.listdir(DATA_DIR))

Fichiers de données : ['.gitkeep', 'checkpoints', 'models', 'products.csv', 'reviews_history.csv']


## 1. SparkSession (le premier lancement télécharge les connecteurs, ~1 min)

In [5]:
spark = get_spark("00-verification")
print("OK - Spark", spark.version)

Spark 3.5.3 prêt - Spark UI : http://localhost:4040
OK - Spark 3.5.3


## 2. Lecture des données générées

In [ ]:
products = spark.read.csv(f"{DATA_DIR}/products.csv", header=True, inferSchema=True)
products.show(5)
print("OK -", products.count(), "produits")

## 3. Connexion PostgreSQL (écriture puis lecture d'une table de test)

In [6]:
write_table(spark.createDataFrame([(1, "ok")], ["id", "status"]), "healthcheck", mode="overwrite")
read_table(spark, "healthcheck").show()
print("OK - PostgreSQL")

+---+------+
| id|status|
+---+------+
|  1|    ok|
+---+------+

OK - PostgreSQL


## 4. Connexion Kafka
Nécessite que le topic `reviews_stream` existe (Partie 1 du sujet). Lecture *batch* des messages déjà présents.

In [7]:
from shopstream_utils import KAFKA_BOOTSTRAP, TOPIC
raw = (spark.read.format("kafka")
       .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
       .option("subscribe", TOPIC)
       .option("startingOffsets", "earliest")
       .load())
raw.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "partition", "offset").show(5, truncate=80)
print("OK - Kafka :", raw.count(), "messages dans le topic")

+-----+--------------------------------------------------------------------------------+---------+------+
|  key|                                                                           value|partition|offset|
+-----+--------------------------------------------------------------------------------+---------+------+
|p0080|{"review_id": "d6807a7f-f725-e365-f640-81bdd93c1c0f", "event_time": "2026-09-...|        0|     0|
|p0080|{"review_id": "9e5ce5ce-9fe2-22c8-77f3-ef20d5343182", "event_time": "2026-09-...|        0|     1|
|p0080|{"review_id": "424cf4e9-7868-c6c5-4d77-96b4433ff1ee", "event_time": "2026-09-...|        0|     2|
|p0080|{"review_id": "9b666f08-e249-71d5-79fa-2f802c67f682", "event_time": "2026-09-...|        0|     3|
|p0140|{"review_id": "c6f10e74-929f-275b-0d06-ef73672b922f", "event_time": "2026-09-...|        0|     4|
+-----+--------------------------------------------------------------------------------+---------+------+
only showing top 5 rows

OK - Kafka : 2170 mes